# Standalone versus official PYTHIA dijet comparison

Compare generator-level dijet distributions produced by the standalone analysis with the corresponding official CMS-style analysis output for CUEP8M1 and CP5. The default pairs are `pythia_jetId.root` versus `cuep8m1_gen_dijet_histograms.root`, and `HiForestMiniAOD_hist.root` versus `cp5_gen_dijet_histograms.root`.

### Physics and statistical conventions

Each stored $\eta_{CM}^{dijet}$ TH2 is projected over the same half-open $p_T^{ave}$ interval and normalized independently according to `NORMALIZATION`; the default unit bin-content sum compares shapes rather than absolute weighted yields. The lower panel is Standalone / Official with ROOT independent-error propagation. The CUEP8M1 official file is the combined pPb-oriented production, whereas the standalone file is pp; that comparison checks the stored analysis outputs and does not make the collision systems equivalent.

Forward/Backward is constructed separately for every input from the **unnormalized** stored Forward and Backward projections. Both F/B construction and the later Standalone/Official ratio use independent-error propagation; binomial division is never used.

## Environment and imports

Start Jupyter from the repository root with `py-env/bin/python -m jupyter notebook`.

In [ ]:
# Cell role: initialize the project Python/PyROOT environment and shared plotting helpers.
# Interpretation: no physics object is changed; ROOT ownership is configured before input is opened.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
from IPython.display import display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'CMakeLists.txt').is_file() and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate jetAnalysis. Start Jupyter from its root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS, STANDARD_DIJET_ETA_CUT_INDEX, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.dijet_closures import DijetClosureCurve, build_dijet_gen_comparisons
from hist_analysis.python.plot_labels import CMS_SIMULATION_HEADER
from hist_analysis.python.plotting import draw_closure
from hist_analysis.python.root_style import PlotStyle, configure_cms_header, configure_mplhep_export

ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration and input resolution

`TUNES` can select either comparison or both. Environment variables override all four input files. Index 5 is the standard stored per-jet acceptance $|\eta_{CM}^{jet}|<1.9$. ROOT PDFs are always saved; `SAVE_MPLHEP` optionally adds parallel mplhep PDFs beneath `mplhep/`, and `SAVE_MPLHEP_PNG` additionally writes PNG copies.

In [ ]:
# Cell role: define user-facing inputs, binning, normalization, and plotting controls.
# Interpretation: these settings select the compared productions and displayed phase space.
TUNES = ('CUEP8M1', 'CP5')
PTAVE_BIN_SET = 'test'  # test or standard
from hist_analysis.config.histograms import DIJET_ETA_CUTS, STANDARD_DIJET_ETA_CUT_INDEX
ETA_CUTS = DIJET_ETA_CUTS  # shared current production cut order
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
REBIN_ETA = 2
NORMALIZATION = 'integral'  # none, integral, or bin_width; never applied before F/B
ETA_Y_RANGE = None
ETA_RATIO_RANGE = (0.75, 1.25)
FB_Y_RANGE = (0.75, 1.25)
FB_RATIO_RANGE = (0.75, 1.25)
DRAW_GRID = True
SAVE_PNG = False
SAVE_MPLHEP = False
SAVE_MPLHEP_PNG = False
CMS_HEADER_LEFT = CMS_SIMULATION_HEADER
CMS_HEADER_RIGHT = '8.16 TeV'
OUTPUT_DIR = Path(os.environ.get('MC_STANDALONE_VS_OFFICIAL_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis' / 'output' / 'mc_standalone_vs_official'))

base = Path.home() / 'cernbox/ana/pPb8160/pythia'
PAIR_FILES = {
    'CUEP8M1': {
        'Official CUEP8M1': Path(os.environ.get('CUEP8M1_OFFICIAL_FILE', base / 'pythia_jetId.root')).expanduser(),
        'Standalone CUEP8M1': Path(os.environ.get('CUEP8M1_STANDALONE_FILE', base / 'cuep8m1tune/cuep8m1_gen_dijet_histograms.root')).expanduser(),
    },
    'CP5': {
        'Official CP5': Path(os.environ.get('CP5_OFFICIAL_FILE', base / 'cp5tune/HiForestMiniAOD_hist.root')).expanduser(),
        'Standalone CP5': Path(os.environ.get('CP5_STANDALONE_FILE', base / 'cp5tune/cp5_gen_dijet_histograms.root')).expanduser(),
    },
}
PAIR_CONTEXT = {
    'CUEP8M1': ('Official: pPb-oriented', 'Standalone: pp'),
    'CP5': ('Official and standalone: pp',),
}
PAIR_COLORS = {'CUEP8M1': ROOT.kBlue + 1, 'CP5': ROOT.kRed + 1}

if not TUNES or len(set(TUNES)) != len(TUNES) or any(tune not in PAIR_FILES for tune in TUNES):
    raise ValueError(f'TUNES must contain unique entries selected from {tuple(PAIR_FILES)}')
if PTAVE_BIN_SET not in {'test', 'standard'}:
    raise ValueError("PTAVE_BIN_SET must be 'test' or 'standard'")
if NORMALIZATION not in {'none', 'integral', 'bin_width'}:
    raise ValueError("NORMALIZATION must be 'none', 'integral', or 'bin_width'")
if ETA_CUT_INDEX not in range(len(ETA_CUTS)):
    raise ValueError(f'ETA_CUT_INDEX must be in [0, {len(ETA_CUTS)-1}]')
if isinstance(REBIN_ETA, bool) or not isinstance(REBIN_ETA, int) or REBIN_ETA < 1:
    raise ValueError('REBIN_ETA must be a positive integer')
for tune in TUNES:
    for label, filename in PAIR_FILES[tune].items():
        if not filename.is_file():
            raise FileNotFoundError(f'Missing {label} ROOT file: {filename}')

PTAVE_BINS = tuple(TEST_DIJET_PTAVE_BINS if PTAVE_BIN_SET == 'test' else DIJET_PTAVE_BINS)
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
ROOT_PLOT_STYLE = PlotStyle(canvas_width=800, canvas_height=800, top_margin=0.12, cms_header_y=0.985)
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_MPLHEP_PNG)
for tune in TUNES:
    print(tune, '; '.join(PAIR_CONTEXT[tune]))
    for label, filename in PAIR_FILES[tune].items():
        print(f'  {label}: {filename}')
print(f'Per-jet eta cut: |etaCM| < {ETA_CUT:g} (index {ETA_CUT_INDEX})')
print(f'pTave bins ({PTAVE_BIN_SET}):', PTAVE_BINS)

## Build projections

The shared dijet helper reads `hGenDijetPtEtaCM_{index}` plus the stored Forward and Backward TH2 histograms, projects identical $p_T^{ave}$ intervals, checks their binning, and constructs F/B with `TH1::Divide(..., '')`. Full distributions may be normalized for display, while F/B always uses the raw weighted projections.

In [ ]:
# Cell role: project the official and standalone generator dijet inputs.
# Interpretation: full eta shapes use the configured display normalization; F/B uses unnormalized yields.
GEN_CURVE = DijetClosureCurve('Gen', 'hGenDijetPtEtaCM_{eta_cut_index}', 'hGenDijetPtEtaForward_{eta_cut_index}', 'hGenDijetPtEtaBackward_{eta_cut_index}')
PROJECTION_CACHE = {}
for tune in TUNES:
    for ptave_range in PTAVE_BINS:
        entry = {'eta_shapes': {}, 'forward_backward': {}, 'keys': {}}
        for sample_index, (label, filename) in enumerate(PAIR_FILES[tune].items()):
            shapes, fb, keys = build_dijet_gen_comparisons(
                filename, (GEN_CURVE,), eta_cut_index=ETA_CUT_INDEX, ptave_range=ptave_range,
                nominal='Gen', rebin_eta=REBIN_ETA, normalization=NORMALIZATION, ratio_option='',
            )
            sample_tag = 'official' if sample_index == 0 else 'standalone'
            shapes['Gen'].SetName(f'{tune.lower()}_{sample_tag}_eta_{ptave_range[0]:g}_{ptave_range[1]:g}')
            fb['Gen'].SetName(f'{tune.lower()}_{sample_tag}_fb_{ptave_range[0]:g}_{ptave_range[1]:g}')
            entry['eta_shapes'][label] = shapes['Gen']
            entry['forward_backward'][label] = fb['Gen']
            entry['keys'][label] = keys['Gen']
        PROJECTION_CACHE[(tune, tuple(ptave_range))] = entry

if ETA_Y_RANGE is None:
    eta_upper = max(h.GetBinContent(i) + h.GetBinError(i) for entry in PROJECTION_CACHE.values() for h in entry['eta_shapes'].values() for i in range(1, h.GetNbinsX() + 1))
    COMMON_ETA_Y_RANGE = (0.0, 1.20 * eta_upper)
else:
    COMMON_ETA_Y_RANGE = ETA_Y_RANGE
print('Common eta-distribution y range:', COMMON_ETA_Y_RANGE)

## Dijet $\eta_{CM}$ shapes and Forward/Backward ratios

For each tune and $p_T^{ave}$ interval, the upper panel overlays official and standalone results. The lower panel shows Standalone / Official. CUEP8M1 uses blue filled/open circles and CP5 uses red filled/open circles.

In [ ]:
# Cell role: draw and save pairwise full-eta and F/B comparisons.
# Interpretation: deviations from unity identify differences between standalone and official stored outputs.
results = {}
y_titles = {'none': 'Weighted dijets', 'integral': '1/N dN/d#eta_{CM}^{dijet}', 'bin_width': '1/N dN/d#eta_{CM}^{dijet}'}
for tune in TUNES:
    official_label, standalone_label = tuple(PAIR_FILES[tune])
    color = PAIR_COLORS[tune]
    curve_styles = {official_label: (color, 20), standalone_label: (color, 24)}
    for low, high in PTAVE_BINS:
        ptave_range = (low, high)
        entry = PROJECTION_CACHE[(tune, ptave_range)]
        tag = f'{tune.lower()}_ptave_{low:g}_{high:g}'.replace('.', 'p')
        annotations = (
            f'PYTHIA {tune}', *PAIR_CONTEXT[tune], 'Gen dijets',
            f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
            f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
            'p_{T}^{Lead/SubLead} > 50/40 GeV', DIJET_DELTA_PHI_SELECTION_LABEL,
        )
        eta_canvas, eta_ratios = draw_closure(
            entry['eta_shapes'], official_label, title='', x_title='#eta_{CM}^{dijet}',
            y_title=y_titles[NORMALIZATION], ratio_range=ETA_RATIO_RANGE,
            ratio_y_title='Standalone / Official', ratio_option='', draw_nominal_ratio=False,
            x_range=(-ETA_CUT - 0.1, ETA_CUT + 0.1), y_range=COMMON_ETA_Y_RANGE,
            grid=DRAW_GRID, annotations=annotations, curve_styles=curve_styles, style=ROOT_PLOT_STYLE,
            cms_header_left=CMS_HEADER_LEFT, cms_header_right=CMS_HEADER_RIGHT,
            output=OUTPUT_DIR / f'{tag}_distribution.pdf', save_png=SAVE_PNG, canvas_name=f'{tag}_distribution',
        )
        fb_canvas, fb_ratios = draw_closure(
            entry['forward_backward'], official_label, title='', x_title='|#eta_{CM}^{dijet}|',
            y_title='Forward / Backward', ratio_range=FB_RATIO_RANGE,
            ratio_y_title='Standalone / Official', ratio_option='', draw_nominal_ratio=False,
            x_range=(0.0, ETA_CUT + 0.1), y_range=FB_Y_RANGE,
            grid=DRAW_GRID, annotations=annotations, curve_styles=curve_styles, style=ROOT_PLOT_STYLE,
            cms_header_left=CMS_HEADER_LEFT, cms_header_right=CMS_HEADER_RIGHT,
            output=OUTPUT_DIR / f'{tag}_forward_backward.pdf', save_png=SAVE_PNG, canvas_name=f'{tag}_forward_backward',
        )
        results[(tune, ptave_range)] = {
            **entry, 'eta_ratio': eta_ratios[standalone_label], 'fb_ratio': fb_ratios[standalone_label],
            'eta_canvas': eta_canvas, 'fb_canvas': fb_canvas,
        }
        print(tune, ptave_range, entry['keys'])
        display(eta_canvas)
        display(fb_canvas)